# 3W Toolkit v3 — Dataset para dados reais não rotulados

Este notebook cria uma versão independente das classes `ParquetDatasetConfig` e `ParquetDataset` do 3W Toolkit para um cenário em que:

- os dados são **não rotulados**;
- todos os arquivos são **dados reais**;
- existem **vários arquivos `.parquet` dentro de uma pasta**, possivelmente em subpastas;
- não é necessário conhecer previamente a quantidade de arquivos;
- cada arquivo representa uma instância/série temporal;
- o carregador mantém os sinais em `DataFrame` e fornece metadados básicos.

A implementação é baseada na organização da classe original do Toolkit, que usa uma configuração com `path`, `columns` e `target_column`, percorre arquivos Parquet e possui `load_file()` e `load_instances_by_variable()`.

**Objetivo:** apontar `DATA_PATH` para a pasta contendo os Parquets reais e executar as células de teste para verificar se os dados estão sendo encontrados, lidos e estruturados corretamente.


## 1. Imports e configuração

Apenas `pandas` e `numpy` são necessários para o carregamento. `pyarrow` é usado como engine do Parquet, como na implementação original do Toolkit.

In [1]:
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional, Sequence

import numpy as np
import pandas as pd


# ============================================================
# CONFIGURAÇÃO PRINCIPAL
# ============================================================

# ALTERE PARA A PASTA QUE CONTÉM OS ARQUIVOS PARQUET
DATA_PATH = Path("/home/bruno.martins/dataset")

# Se None, todas as colunas serão carregadas como sinais.
# Se quiser selecionar sensores específicos, use por exemplo:
# SIGNAL_COLUMNS = ["P-PDG", "P-TPT", "T-TPT"]
SIGNAL_COLUMNS: Optional[list[str]] = None

# Se True, procura Parquets também dentro de subpastas.
RECURSIVE = False

# Quantos arquivos serão mostrados no teste inicial.
N_FILES_TO_TEST = 5


## 2. Estruturas de saída

A versão original retorna um objeto `DatasetOutputs` contendo `signal`, `label` e `metadata`.

Como esta versão não depende dos módulos internos do Toolkit, usamos uma pequena estrutura equivalente. Para dados não rotulados, `label` será sempre `None`.

In [2]:
@dataclass
class DatasetOutputs:
    """Saída de uma instância do dataset não rotulado."""

    signal: pd.DataFrame
    label: None = None
    metadata: dict = field(default_factory=dict)


@dataclass
class UnlabeledParquetDatasetConfig:
    """Configuração do dataset real não rotulado."""

    path: Path | str
    columns: Optional[Sequence[str]] = None
    recursive: bool = True
    parquet_engine: str = "pyarrow"

    def __post_init__(self):
        self.path = Path(self.path)


## 3. Classe `UnlabeledParquetDataset`

Principais diferenças em relação à classe original:

1. não tenta fazer download ou extrair uma versão oficial do 3W;
2. não exige uma quantidade fixa de Parquets;
3. não exige pastas numéricas de classes;
4. não procura uma coluna `class`;
5. todos os arquivos são considerados **reais e não rotulados**;
6. `label=None` para todas as instâncias;
7. o nome/caminho do arquivo é preservado nos metadados;
8. mantém `load_instances_by_variable()` para facilitar o uso posterior em modelos e clustering.

A classe original também carrega um único Parquet em `load_file()` e pode agrupar as instâncias por variável em `load_instances_by_variable()`.

In [3]:
class UnlabeledParquetDataset:
    """Dataset para múltiplos Parquets reais e não rotulados."""

    def __init__(self, config: UnlabeledParquetDatasetConfig):
        self.config = config
        self.root = Path(config.path)

        if not self.root.exists():
            raise FileNotFoundError(
                f"Pasta do dataset não encontrada: {self.root.resolve()}"
            )

        if not self.root.is_dir():
            raise NotADirectoryError(
                f"DATA_PATH precisa ser uma pasta: {self.root.resolve()}"
            )

        # Descobre todos os Parquets disponíveis.
        pattern = "**/*.parquet" if config.recursive else "*.parquet"
        self.files_events = sorted(self.root.glob(pattern))

        if not self.files_events:
            raise FileNotFoundError(
                f"Nenhum arquivo .parquet encontrado em: {self.root.resolve()}"
            )

        # Remove duplicatas caso algum caminho seja repetido.
        self.files_events = list(dict.fromkeys(self.files_events))

    def __len__(self) -> int:
        return len(self.files_events)

    def __getitem__(self, idx: int) -> DatasetOutputs:
        return self.load_file(idx)

    def _relative_path(self, path: Path) -> str:
        return str(path.relative_to(self.root))

    def load_file(self, idx: int) -> DatasetOutputs:
        """Carrega uma instância Parquet sem tentar obter rótulos."""

        if not isinstance(idx, (int, np.integer)):
            raise TypeError("idx precisa ser um inteiro.")

        if idx < 0 or idx >= len(self):
            raise IndexError(
                f"Índice {idx} fora do intervalo [0, {len(self) - 1}]."
            )

        file_path = self.files_events[idx]

        df = pd.read_parquet(
            file_path,
            engine=self.config.parquet_engine,
        )

        if not isinstance(df, pd.DataFrame):
            raise TypeError(
                f"O arquivo {file_path} não foi carregado como DataFrame."
            )

        # Seleção opcional de sensores/variáveis.
        if self.config.columns is not None:
            requested = list(self.config.columns)
            missing = [c for c in requested if c not in df.columns]

            if missing:
                raise ValueError(
                    f"Colunas ausentes em {self._relative_path(file_path)}: "
                    f"{missing}"
                )

            signal_df = df.loc[:, requested].copy()
        else:
            signal_df = df.copy()

        metadata = {
            "file_name": file_path.name,
            "relative_path": self._relative_path(file_path),
            "absolute_path": str(file_path.resolve()),
            "event_type": "real",
            "event_class": None,
            "labeled": False,
            "n_rows": len(signal_df),
            "n_columns": len(signal_df.columns),
            "columns": signal_df.columns.tolist(),
        }

        return DatasetOutputs(
            signal=signal_df,
            label=None,
            metadata=metadata,
        )

    def get_file_list(self) -> list[Path]:
        """Retorna a lista dos arquivos Parquet encontrados."""
        return self.files_events.copy()

    def schema_summary(self, max_files: Optional[int] = None) -> pd.DataFrame:
        """Resume nome, tamanho, número de linhas e colunas dos arquivos."""

        files = self.files_events
        if max_files is not None:
            files = files[:max_files]

        rows = []

        for path in files:
            df = pd.read_parquet(path, engine=self.config.parquet_engine)

            rows.append({
                "file_name": path.name,
                "relative_path": self._relative_path(path),
                "size_MB": path.stat().st_size / (1024 ** 2),
                "n_rows": len(df),
                "n_columns": len(df.columns),
                "columns": ", ".join(map(str, df.columns)),
                "dtypes": ", ".join(
                    f"{c}:{df[c].dtype}" for c in df.columns
                ),
            })

        return pd.DataFrame(rows)

    def validate_files(
        self,
        max_files: Optional[int] = None,
        require_same_columns: bool = False,
        allow_empty: bool = False,
    ) -> pd.DataFrame:
        """Valida se os Parquets podem ser lidos e têm estrutura coerente."""

        files = self.files_events
        if max_files is not None:
            files = files[:max_files]

        results = []

        reference_columns = None

        for path in files:
            result = {
                "file_name": path.name,
                "relative_path": self._relative_path(path),
                "read_ok": False,
                "non_empty": False,
                "n_rows": 0,
                "n_columns": 0,
                "has_nan": False,
                "same_columns": True,
                "error": None,
            }

            try:
                df = pd.read_parquet(
                    path,
                    engine=self.config.parquet_engine,
                )

                result["read_ok"] = True
                result["n_rows"] = len(df)
                result["n_columns"] = len(df.columns)
                result["non_empty"] = len(df) > 0
                result["has_nan"] = bool(df.isna().any().any())

                current_columns = list(df.columns)

                if reference_columns is None:
                    reference_columns = current_columns
                else:
                    result["same_columns"] = (
                        current_columns == reference_columns
                    )

                if not allow_empty and len(df) == 0:
                    result["error"] = "Arquivo vazio."

                if (
                    require_same_columns
                    and not result["same_columns"]
                ):
                    result["error"] = "Colunas diferentes do primeiro arquivo."

            except Exception as exc:
                result["error"] = f"{type(exc).__name__}: {exc}"

            results.append(result)

        return pd.DataFrame(results)

    def load_instances_by_variable(
        self,
        variables: Optional[list[str]] = None,
    ) -> dict[str, list[np.ndarray]]:
        """Retorna {variável: [array_da_instância, ...]}.

        Arquivos que não possuem uma variável solicitada são ignorados para
        aquela variável.
        """

        if variables is None:
            if self.config.columns is None:
                # Descobre as colunas a partir do primeiro arquivo.
                first = self.load_file(0).signal
                variables = first.columns.tolist()
            else:
                variables = list(self.config.columns)

        data_map: dict[str, list[np.ndarray]] = {
            var: [] for var in variables
        }

        for idx in range(len(self)):
            signal_df = self.load_file(idx).signal

            for var in variables:
                if var in signal_df.columns:
                    values = signal_df[var].to_numpy()
                    data_map[var].append(values)

        return data_map

    def describe(self, max_files: Optional[int] = None) -> pd.DataFrame:
        """Estatísticas básicas das variáveis numéricas."""

        files = self.files_events
        if max_files is not None:
            files = files[:max_files]

        rows = []

        for path in files:
            df = pd.read_parquet(
                path,
                engine=self.config.parquet_engine,
            )

            numeric = df.select_dtypes(include=np.number)

            for column in numeric.columns:
                values = numeric[column].to_numpy(dtype=float)

                rows.append({
                    "file_name": path.name,
                    "variable": column,
                    "n": len(values),
                    "min": np.nanmin(values) if len(values) else np.nan,
                    "max": np.nanmax(values) if len(values) else np.nan,
                    "mean": np.nanmean(values) if len(values) else np.nan,
                    "std": np.nanstd(values) if len(values) else np.nan,
                    "nan_count": int(np.isnan(values).sum()),
                })

        return pd.DataFrame(rows)


## 4. Criar o dataset

In [4]:
config = UnlabeledParquetDatasetConfig(
    path=DATA_PATH,
    columns=SIGNAL_COLUMNS,
    recursive=RECURSIVE,
)

dataset = UnlabeledParquetDataset(config)

print("Dataset criado com sucesso!")
print(f"Pasta: {dataset.root.resolve()}")
print(f"Quantidade de arquivos Parquet: {len(dataset)}")


Dataset criado com sucesso!
Pasta: /nfs/home/bruno.martins/dataset
Quantidade de arquivos Parquet: 58


## 5. Teste 1 — arquivos encontrados

In [5]:
files = dataset.get_file_list()

print(f"Total de arquivos: {len(files)}")
print("\nPrimeiros arquivos:")

for path in files[:N_FILES_TO_TEST]:
    print(" -", path.relative_to(dataset.root))

assert len(files) > 0, "Nenhum arquivo Parquet foi encontrado."

print("\n[OK] Os arquivos Parquet foram encontrados.")


Total de arquivos: 58

Primeiros arquivos:
 - WELL-00005_20170331050014.parquet
 - WELL-00007_20170517200012.parquet
 - WELL-00008_20170610210246.parquet
 - WELL-00009_20170313150804.parquet
 - WELL-00011_20140515083000.parquet

[OK] Os arquivos Parquet foram encontrados.


## 6. Teste 2 — leitura dos primeiros arquivos

Este teste verifica:

- se cada Parquet pode ser aberto;
- se o resultado é um `DataFrame`;
- se possui linhas;
- quais colunas existem;
- se a saída é realmente não rotulada (`label is None`);
- se os metadados identificam o arquivo como `real`.

In [6]:
n_test = min(N_FILES_TO_TEST, len(dataset))

for idx in range(n_test):
    item = dataset[idx]

    print("=" * 80)
    print(f"Índice: {idx}")
    print(f"Arquivo: {item.metadata['relative_path']}")
    print(f"Tipo: {item.metadata['event_type']}")
    print(f"Rotulado: {item.metadata['labeled']}")
    print(f"Shape: {item.signal.shape}")
    print(f"Label: {item.label}")
    print(f"Colunas: {item.signal.columns.tolist()}")

    assert isinstance(item.signal, pd.DataFrame)
    assert item.label is None
    assert item.metadata["event_type"] == "real"
    assert item.metadata["labeled"] is False
    assert len(item.signal) > 0
    assert item.signal.columns.tolist() == item.metadata["columns"]

print("\n[OK] Os arquivos foram lidos corretamente.")


Índice: 0
Arquivo: WELL-00005_20170331050014.parquet
Tipo: real
Rotulado: False
Shape: (235318, 27)
Label: None
Colunas: ['ABER-CKGL', 'ABER-CKP', 'ESTADO-DHSV', 'ESTADO-M1', 'ESTADO-M2', 'ESTADO-PXO', 'ESTADO-SDV-GL', 'ESTADO-SDV-P', 'ESTADO-W1', 'ESTADO-W2', 'ESTADO-XO', 'P-ANULAR', 'P-JUS-BS', 'P-JUS-CKGL', 'P-JUS-CKP', 'P-MON-CKGL', 'P-MON-CKP', 'P-MON-SDV-P', 'P-PDG', 'PT-P', 'P-TPT', 'QBS', 'QGL', 'T-JUS-CKP', 'T-MON-CKP', 'T-PDG', 'T-TPT']
Índice: 1
Arquivo: WELL-00007_20170517200012.parquet
Tipo: real
Rotulado: False
Shape: (109560, 27)
Label: None
Colunas: ['ABER-CKGL', 'ABER-CKP', 'ESTADO-DHSV', 'ESTADO-M1', 'ESTADO-M2', 'ESTADO-PXO', 'ESTADO-SDV-GL', 'ESTADO-SDV-P', 'ESTADO-W1', 'ESTADO-W2', 'ESTADO-XO', 'P-ANULAR', 'P-JUS-BS', 'P-JUS-CKGL', 'P-JUS-CKP', 'P-MON-CKGL', 'P-MON-CKP', 'P-MON-SDV-P', 'P-PDG', 'PT-P', 'P-TPT', 'QBS', 'QGL', 'T-JUS-CKP', 'T-MON-CKP', 'T-PDG', 'T-TPT']
Índice: 2
Arquivo: WELL-00008_20170610210246.parquet
Tipo: real
Rotulado: False
Shape: (137994, 27

## 7. Teste 3 — validação de todos os arquivos

In [7]:
validation = dataset.validate_files(
    require_same_columns=False,
    allow_empty=False,
)

display(validation)

failed = validation[validation["read_ok"] == False]

if not failed.empty:
    print("\nArquivos com erro:")
    display(failed)

assert failed.empty, (
    f"{len(failed)} arquivo(s) não puderam ser carregados."
)

empty = validation[validation["non_empty"] == False]

assert empty.empty, (
    f"{len(empty)} arquivo(s) estão vazios."
)

print(
    f"\n[OK] {len(validation)} arquivo(s) foram validados "
    "sem erro de leitura."
)


,file_name,relative_path,read_ok,non_empty,n_rows,n_columns,has_nan,same_columns,error
0,WELL-00005_20170331050014.parquet,WELL-00005_20170331050014.parquet,True,True,235318,27,True,True,None
1,WELL-00007_20170517200012.parquet,WELL-00007_20170517200012.parquet,True,True,109560,27,True,True,None
2,WELL-00008_20170610210246.parquet,WELL-00008_20170610210246.parquet,True,True,137994,27,True,True,None
3,WELL-00009_20170313150804.parquet,WELL-00009_20170313150804.parquet,True,True,173,27,True,True,None
4,WELL-00011_20140515083000.parquet,WELL-00011_20140515083000.parquet,True,True,207886,27,True,True,None
5,WELL-00012_20170320011000.parquet,WELL-00012_20170320011000.parquet,True,True,858,27,True,True,None
6,WELL-00013_20170329010229.parquet,WELL-00013_20170329010229.parquet,True,True,231,27,True,True,None
7,WELL-00022_20180802233838.parquet,WELL-00022_20180802233838.parquet,True,True,132742,27,True,True,None
8,WELL-00023_20180826212652.parquet,WELL-00023_20180826212652.parquet,True,True,96393,27,True,True,None
9,WELL-00024_20160704180000.parquet,WELL-00024_20160704180000.parquet,True,True,166450,27,True,True,None



[OK] 58 arquivo(s) foram validados sem erro de leitura.


## 8. Verificar se todos os arquivos possuem o mesmo esquema

Não assumimos automaticamente que todos os arquivos têm exatamente as mesmas colunas. Primeiro mostramos o resultado. Se o seu conjunto real tiver um esquema fixo, `require_same_columns=True` pode ser usado para transformar essa condição em uma validação obrigatória.

In [8]:
validation_schema = dataset.validate_files(
    require_same_columns=True,
    allow_empty=False,
)

display(validation_schema)

schema_errors = validation_schema[
    validation_schema["same_columns"] == False
]

if schema_errors.empty:
    print("[OK] Todos os arquivos possuem as mesmas colunas.")
else:
    print(
        f"[ATENÇÃO] {len(schema_errors)} arquivo(s) possuem "
        "colunas diferentes do primeiro arquivo."
    )
    display(schema_errors[[
        "file_name",
        "relative_path",
        "same_columns",
        "error",
    ]])


,file_name,relative_path,read_ok,non_empty,n_rows,n_columns,has_nan,same_columns,error
0,WELL-00005_20170331050014.parquet,WELL-00005_20170331050014.parquet,True,True,235318,27,True,True,None
1,WELL-00007_20170517200012.parquet,WELL-00007_20170517200012.parquet,True,True,109560,27,True,True,None
2,WELL-00008_20170610210246.parquet,WELL-00008_20170610210246.parquet,True,True,137994,27,True,True,None
3,WELL-00009_20170313150804.parquet,WELL-00009_20170313150804.parquet,True,True,173,27,True,True,None
4,WELL-00011_20140515083000.parquet,WELL-00011_20140515083000.parquet,True,True,207886,27,True,True,None
5,WELL-00012_20170320011000.parquet,WELL-00012_20170320011000.parquet,True,True,858,27,True,True,None
6,WELL-00013_20170329010229.parquet,WELL-00013_20170329010229.parquet,True,True,231,27,True,True,None
7,WELL-00022_20180802233838.parquet,WELL-00022_20180802233838.parquet,True,True,132742,27,True,True,None
8,WELL-00023_20180826212652.parquet,WELL-00023_20180826212652.parquet,True,True,96393,27,True,True,None
9,WELL-00024_20160704180000.parquet,WELL-00024_20160704180000.parquet,True,True,166450,27,True,True,None


[OK] Todos os arquivos possuem as mesmas colunas.


## 9. Inspecionar o conteúdo de uma instância

In [9]:
sample = dataset[0]

print("Metadados:")
for key, value in sample.metadata.items():
    print(f"  {key}: {value}")

print("\nPrimeiras linhas:")
display(sample.signal.head())

print("\nTipos:")
display(sample.signal.dtypes)

print("\nValores ausentes por coluna:")
display(sample.signal.isna().sum())


Metadados:
  file_name: WELL-00005_20170331050014.parquet
  relative_path: WELL-00005_20170331050014.parquet
  absolute_path: /nfs/home/bruno.martins/dataset/WELL-00005_20170331050014.parquet
  event_type: real
  event_class: None
  labeled: False
  n_rows: 235318
  n_columns: 27
  columns: ['ABER-CKGL', 'ABER-CKP', 'ESTADO-DHSV', 'ESTADO-M1', 'ESTADO-M2', 'ESTADO-PXO', 'ESTADO-SDV-GL', 'ESTADO-SDV-P', 'ESTADO-W1', 'ESTADO-W2', 'ESTADO-XO', 'P-ANULAR', 'P-JUS-BS', 'P-JUS-CKGL', 'P-JUS-CKP', 'P-MON-CKGL', 'P-MON-CKP', 'P-MON-SDV-P', 'P-PDG', 'PT-P', 'P-TPT', 'QBS', 'QGL', 'T-JUS-CKP', 'T-MON-CKP', 'T-PDG', 'T-TPT']

Primeiras linhas:


,ABER-CKGL,ABER-CKP,ESTADO-DHSV,ESTADO-M1,ESTADO-M2,ESTADO-PXO,ESTADO-SDV-GL,ESTADO-SDV-P,ESTADO-W1,ESTADO-W2,...,P-MON-SDV-P,P-PDG,PT-P,P-TPT,QBS,QGL,T-JUS-CKP,T-MON-CKP,T-PDG,T-TPT
timestamp,,,,,,,,,,,,,,,,,,,,,
2017-03-31 05:00:14,0.0,28.371908,0.0,1.0,1.0,0.0,NaN,1.0,1.0,0.0,...,NaN,NaN,NaN,2.079164e+07,NaN,NaN,67.509010,NaN,NaN,106.372238
2017-03-31 05:01:14,0.0,28.372513,0.0,1.0,1.0,0.0,NaN,1.0,1.0,0.0,...,NaN,NaN,NaN,2.079373e+07,NaN,NaN,67.507919,NaN,NaN,106.369843
2017-03-31 05:02:14,0.0,28.373117,0.0,1.0,1.0,0.0,NaN,1.0,1.0,0.0,...,NaN,NaN,NaN,2.079185e+07,NaN,NaN,67.506828,NaN,NaN,106.368790
2017-03-31 05:03:14,0.0,28.373722,0.0,1.0,1.0,0.0,NaN,1.0,1.0,0.0,...,NaN,NaN,NaN,2.079457e+07,NaN,NaN,67.505737,NaN,NaN,106.369843
2017-03-31 05:04:14,0.0,28.374325,0.0,1.0,1.0,0.0,NaN,1.0,1.0,0.0,...,NaN,NaN,NaN,2.079499e+07,NaN,NaN,67.504646,NaN,NaN,106.365341



Tipos:


ABER-CKGL        float64
ABER-CKP         float64
ESTADO-DHSV      float64
ESTADO-M1        float64
ESTADO-M2        float64
ESTADO-PXO       float64
ESTADO-SDV-GL    float64
ESTADO-SDV-P     float64
ESTADO-W1        float64
ESTADO-W2        float64
ESTADO-XO        float64
P-ANULAR         float64
P-JUS-BS         float64
P-JUS-CKGL       float64
P-JUS-CKP        float64
P-MON-CKGL       float64
P-MON-CKP        float64
P-MON-SDV-P      float64
P-PDG            float64
PT-P             float64
P-TPT            float64
QBS              float64
QGL              float64
T-JUS-CKP        float64
T-MON-CKP        float64
T-PDG            float64
T-TPT            float64
dtype: object


Valores ausentes por coluna:


ABER-CKGL            42
ABER-CKP             45
ESTADO-DHSV          46
ESTADO-M1            44
ESTADO-M2            48
ESTADO-PXO           44
ESTADO-SDV-GL    235318
ESTADO-SDV-P         47
ESTADO-W1            47
ESTADO-W2            46
ESTADO-XO            48
P-ANULAR             41
P-JUS-BS         235318
P-JUS-CKGL           43
P-JUS-CKP        235318
P-MON-CKGL       235318
P-MON-CKP            42
P-MON-SDV-P      235318
P-PDG            235318
PT-P             235318
P-TPT                43
QBS              235318
QGL              235318
T-JUS-CKP            37
T-MON-CKP        235318
T-PDG            235318
T-TPT                42
dtype: int64

## 10. Teste de `load_instances_by_variable()`

Essa função preserva a mesma ideia da classe original: para cada variável, construir uma lista em que cada elemento corresponde a uma instância/arquivo. 

Isso é útil posteriormente para alimentar rotinas de clustering, autoencoders, CNN/LSTM/GRU ou outros modelos de séries temporais.

In [10]:
# Se SIGNAL_COLUMNS for None, as colunas do primeiro arquivo serão usadas.
variables = (
    SIGNAL_COLUMNS
    if SIGNAL_COLUMNS is not None
    else dataset.load_file(0).signal.columns.tolist()
)

data_by_variable = dataset.load_instances_by_variable(
    variables=list(variables)
)

print("Variáveis carregadas:")

for variable, instances in data_by_variable.items():
    print(
        f" - {variable}: "
        f"{len(instances)} instâncias; "
        f"primeiro shape = "
        f"{instances[0].shape if instances else None}"
    )

assert len(data_by_variable) > 0

for variable, instances in data_by_variable.items():
    assert isinstance(instances, list)
    for values in instances:
        assert isinstance(values, np.ndarray)
        assert values.ndim == 1

print("\n[OK] Dados agrupados corretamente por variável.")


Variáveis carregadas:
 - ABER-CKGL: 58 instâncias; primeiro shape = (235318,)
 - ABER-CKP: 58 instâncias; primeiro shape = (235318,)
 - ESTADO-DHSV: 58 instâncias; primeiro shape = (235318,)
 - ESTADO-M1: 58 instâncias; primeiro shape = (235318,)
 - ESTADO-M2: 58 instâncias; primeiro shape = (235318,)
 - ESTADO-PXO: 58 instâncias; primeiro shape = (235318,)
 - ESTADO-SDV-GL: 58 instâncias; primeiro shape = (235318,)
 - ESTADO-SDV-P: 58 instâncias; primeiro shape = (235318,)
 - ESTADO-W1: 58 instâncias; primeiro shape = (235318,)
 - ESTADO-W2: 58 instâncias; primeiro shape = (235318,)
 - ESTADO-XO: 58 instâncias; primeiro shape = (235318,)
 - P-ANULAR: 58 instâncias; primeiro shape = (235318,)
 - P-JUS-BS: 58 instâncias; primeiro shape = (235318,)
 - P-JUS-CKGL: 58 instâncias; primeiro shape = (235318,)
 - P-JUS-CKP: 58 instâncias; primeiro shape = (235318,)
 - P-MON-CKGL: 58 instâncias; primeiro shape = (235318,)
 - P-MON-CKP: 58 instâncias; primeiro shape = (235318,)
 - P-MON-SDV-P: 5

## 11. Estatísticas básicas dos sinais

Esta etapa não é obrigatória para o carregamento, mas ajuda a identificar rapidamente problemas como:

- Parquets vazios;
- variáveis completamente ausentes;
- quantidade elevada de `NaN`;
- valores constantes;
- escalas inesperadas.

In [11]:
stats = dataset.describe(max_files=N_FILES_TO_TEST)

if stats.empty:
    print("Nenhuma coluna numérica foi encontrada.")
else:
    display(stats.head(50))


/tmp/ipykernel_2667166/2369789932.py:247: RuntimeWarning: All-NaN slice encountered
  "min": np.nanmin(values) if len(values) else np.nan,
/tmp/ipykernel_2667166/2369789932.py:248: RuntimeWarning: All-NaN slice encountered
  "max": np.nanmax(values) if len(values) else np.nan,
/tmp/ipykernel_2667166/2369789932.py:249: RuntimeWarning: Mean of empty slice
  "mean": np.nanmean(values) if len(values) else np.nan,
/home/bruno.martins/miniconda3/envs/3W/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1997: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipykernel_2667166/2369789932.py:247: RuntimeWarning: All-NaN slice encountered
  "min": np.nanmin(values) if len(values) else np.nan,
/tmp/ipykernel_2667166/2369789932.py:248: RuntimeWarning: All-NaN slice encountered
  "max": np.nanmax(values) if len(values) else np.nan,
/tmp/ipykernel_2667166/2369789932.py:249: RuntimeWarning: Mean of empty slice
  "mean": np

,file_name,variable,n,min,max,mean,std,nan_count
0,WELL-00005_20170331050014.parquet,ABER-CKGL,235318,0.000000e+00,1.000000e+02,1.564905e+01,3.626315e+01,42
1,WELL-00005_20170331050014.parquet,ABER-CKP,235318,-2.500000e+01,1.092230e+02,2.470440e+01,7.030432e+00,45
2,WELL-00005_20170331050014.parquet,ESTADO-DHSV,235318,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,46
3,WELL-00005_20170331050014.parquet,ESTADO-M1,235318,0.000000e+00,1.000000e+00,8.590452e-01,3.479749e-01,44
4,WELL-00005_20170331050014.parquet,ESTADO-M2,235318,0.000000e+00,1.000000e+00,6.838186e-01,4.649847e-01,48
5,WELL-00005_20170331050014.parquet,ESTADO-PXO,235318,0.000000e+00,1.000000e+00,1.933915e-03,4.393376e-02,44
6,WELL-00005_20170331050014.parquet,ESTADO-SDV-GL,235318,NaN,NaN,NaN,NaN,235318
7,WELL-00005_20170331050014.parquet,ESTADO-SDV-P,235318,0.000000e+00,1.000000e+00,8.570797e-01,3.499915e-01,47
8,WELL-00005_20170331050014.parquet,ESTADO-W1,235318,0.000000e+00,1.000000e+00,8.632598e-01,3.435729e-01,47
9,WELL-00005_20170331050014.parquet,ESTADO-W2,235318,0.000000e+00,1.000000e+00,6.103574e-02,2.393959e-01,46


## 12. Teste final automatizado

Se esta célula terminar sem `AssertionError`, temos as verificações básicas:

1. a pasta existe;
2. há arquivos Parquet;
3. os arquivos podem ser lidos;
4. eles não estão vazios;
5. cada instância retorna um `DataFrame`;
6. nenhuma instância possui label;
7. todos os arquivos são tratados como `real`;
8. o agrupamento por variável funciona.

In [12]:
# Teste final
assert dataset.root.exists()
assert len(dataset) > 0

for idx in range(len(dataset)):
    item = dataset[idx]

    assert isinstance(item.signal, pd.DataFrame)
    assert item.label is None
    assert item.metadata["event_type"] == "real"
    assert item.metadata["event_class"] is None
    assert item.metadata["labeled"] is False
    assert len(item.signal) > 0

print("==============================================")
print(" TESTE FINAL: PASSOU")
print("==============================================")
print(f"Arquivos testados: {len(dataset)}")
print("Todos os arquivos são tratados como reais.")
print("Todos os arquivos são tratados como não rotulados.")
print("Os Parquets foram carregados corretamente.")


 TESTE FINAL: PASSOU
Arquivos testados: 58
Todos os arquivos são tratados como reais.
Todos os arquivos são tratados como não rotulados.
Os Parquets foram carregados corretamente.


## 13. Como usar com a base real

Substitua somente:

```python
DATA_PATH = Path("/caminho/para/sua/pasta")
```

Se quiser carregar apenas determinados sensores:

```python
SIGNAL_COLUMNS = ["P-PDG", "P-TPT", "T-TPT"]
```

Se os Parquets estiverem diretamente na pasta, sem subpastas:

```python
RECURSIVE = False
```

Caso contrário, mantenha:

```python
RECURSIVE = True
```

A quantidade de arquivos **não precisa ser conhecida antecipadamente**: ela é descoberta automaticamente com `Path.glob()`/`rglob()`.
